# Amharic Capability Distillation

This is the **single master notebook** for the complete response-level knowledge-distillation experiment.

Pipeline:

1. Environment and reproducibility setup
2. 60-prompt evaluation set and leakage firewall
3. Public-data filtering and candidate construction
4. Final 2,800-prompt training pool
5. Gemini teacher-response generation
6. Final teacher-response dataset (1,144 examples)
7. Qwen2.5-0.5B-Instruct + LoRA fine-tuning
8. Base vs distilled evaluation on the same 60 held-out prompts
9. Result analysis and submission artifacts

**Important:** the evaluation prompts are never used as training examples. The code removes exact and near-duplicate training prompts before teacher generation.


## 1. Environment and project configuration

The experiment was run in Google Colab. The training run used a **Tesla T4** GPU.

The project files are kept in Google Drive so that Colab runtime resets do not destroy the datasets or LoRA adapter.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import random
import re
import unicodedata
import pandas as pd
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/drive/MyDrive/amha-distill")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
ADAPTER_DIR = PROJECT_DIR / "adapter"

for folder in [DATA_DIR, OUTPUT_DIR, ADAPTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
print("Python project:", PROJECT_DIR)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.version.cuda)
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GiB"
    )
else:
    print("No GPU currently available.")


In [ ]:
!pip install -q datasets peft transformers accelerate google-genai

## 2. Evaluation set and leakage firewall

The assignment requires **60 held-out evaluation prompts**: 15 for each task.

The central isolation rule is:

> A prompt that is in the evaluation set, or a near-copy of it, must not appear in the training prompt pool.

We therefore normalize text before comparison. Unicode NFC normalization makes equivalent Unicode representations comparable, while whitespace normalization prevents harmless formatting differences from defeating the duplicate check.


In [ ]:
# AMHADISTILL; PUBLIC DATA RECONSTRUCTION + FILTERING

import os
import re
import unicodedata
import pandas as pd
from datasets import load_dataset

# Paths

PROJECT_DIR = "/content/drive/MyDrive/amha-distill"
DATA_DIR = f"{PROJECT_DIR}/data"

os.makedirs(DATA_DIR, exist_ok=True)

SEED = 42

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)


# Load evaluation prompts

eval_path = f"{DATA_DIR}/eval_prompts_60.csv"

eval_df = pd.read_csv(eval_path)

print("\nEvaluation prompts:", len(eval_df))
print("Evaluation tasks:")
print(eval_df["task"].value_counts())


# Normalization function

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


eval_df["normalized_prompt"] = eval_df["prompt"].apply(normalize_text)

EVAL_PROMPTS = set(eval_df["normalized_prompt"])


# Load EthioNLP dataset

print("\nLoading EthioNLP dataset...")

walia = load_dataset(
    "EthioNLP/Amharic_Instruction_dataset",
    split="train"
)

walia_df = walia.to_pandas()

print("Raw rows:", len(walia_df))


# Clean source columns

source_df = walia_df.copy()

for column in ["instruction", "input", "output", "datasource"]:
    if column not in source_df.columns:
        raise ValueError(f"Missing expected column: {column}")

    source_df[column] = (
        source_df[column]
        .fillna("")
        .astype(str)
    )

source_df["source_prompt"] = source_df["instruction"].str.strip()

has_input = source_df["input"].str.strip() != ""

source_df.loc[has_input, "source_prompt"] = (
    source_df.loc[has_input, "source_prompt"]
    + "\n\n"
    + source_df.loc[has_input, "input"].str.strip()
)

source_df["normalized_prompt"] = (
    source_df["source_prompt"]
    .apply(normalize_text)
)


# Remove exact evaluation duplicates

source_df["is_eval_duplicate"] = (
    source_df["normalized_prompt"].isin(EVAL_PROMPTS)
)

eval_duplicates = source_df[
    source_df["is_eval_duplicate"]
].copy()

source_clean_df = source_df[
    ~source_df["is_eval_duplicate"]
].copy()

print("\nEvaluation duplicates removed:",
      len(eval_duplicates))

print("Remaining source rows:",
      len(source_clean_df))


# Build public candidate pool

PUBLIC_SOURCES = {
    "instruction_following": [
        "afrisent",
        "masakhanews"
    ],

    "translation": [
        "amharic_mt amh-eng",
        "amharic_mt eng-amh"
    ],

    "summarization": [
        "xlsum_summerization",
        "xlsum_reverse_summerization",
        "masakhanews"
    ],

    "closed_book_knowledge": [
        "amharicqa"
    ],
}

public_candidates = []

for task, sources in PUBLIC_SOURCES.items():

    subset = source_clean_df[
        source_clean_df["datasource"].isin(sources)
    ].copy()

    for _, row in subset.iterrows():

        prompt = row["source_prompt"].strip()

        if not prompt:
            continue

        public_candidates.append({
            "prompt": prompt,
            "task": task,
            "source": f"EthioNLP:{row['datasource']}",
            "source_output": row["output"].strip()
        })

public_candidates_df = pd.DataFrame(
    public_candidates
)

print("\nPublic candidate pool:",
      len(public_candidates_df))


# SUMMARIZATION FILTER

summary_source_df = public_candidates_df[
    public_candidates_df["source"]
    == "EthioNLP:xlsum_summerization"
].copy()

summary_source_df["prompt_clean"] = (
    summary_source_df["prompt"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

summary_patterns = (
    r"(ማጠቃለያ|"
    r"ማጠቃለል|"
    r"አጠቃል|"
    r"በአጭሩ|"
    r"ዋና\s*ሀሳብ|"
    r"ጠቅላላ\s*ሀሳብ|"
    r"ስለ\s*ምን.*ያወራ|"
    r"ስለ\s*ምን.*የሚያወራ|"
    r"ምን\s*ይላል)"
)

bad_summary_patterns = (
    r"(በአንድ\s*ቃል|"
    r"አንድ\s*ቃል|"
    r"መደብ|"
    r"ምደባ|"
    r"ምን\s*አይነት\s*ይዘት)"
)

is_summary = summary_source_df[
    "prompt_clean"
].str.contains(
    summary_patterns,
    regex=True,
    na=False
)

is_bad_summary = summary_source_df[
    "prompt_clean"
].str.contains(
    bad_summary_patterns,
    regex=True,
    na=False
)

summary_candidates_df = summary_source_df[
    is_summary & ~is_bad_summary
].copy()

summary_candidates_df = (
    summary_candidates_df
    .drop_duplicates(subset=["prompt_clean"])
    .reset_index(drop=True)
)

summary_candidates_df[
    ["prompt", "task", "source", "source_output"]
].to_csv(
    f"{DATA_DIR}/summarization_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)


# TRANSLATION FILTER

translation_source_df = public_candidates_df[
    public_candidates_df["task"] == "translation"
].copy()

translation_source_df["prompt_clean"] = (
    translation_source_df["prompt"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

translation_patterns = (
    r"(ተርጉም|"
    r"ቀይር|"
    r"ምን\s*ማለት\s*ነው|"
    r"እንዴት\s*ሊፃፍ\s*ይችላል)"
)

bad_translation_patterns = (
    r"(ተቀራራቢ|"
    r"ተመሳሳይ|"
    r"ተመሳሳይ\s*አረፍተ|"
    r"similar|"
    r"write\s+a\s+similar|"
    r"ፃፍ\s*$)"
)

prompt_lower = (
    translation_source_df["prompt_clean"]
    .str.lower()
)

is_translation = prompt_lower.str.contains(
    translation_patterns,
    regex=True,
    na=False
)

is_bad_translation = prompt_lower.str.contains(
    bad_translation_patterns,
    regex=True,
    na=False
)

translation_candidates_df = translation_source_df[
    is_translation & ~is_bad_translation
].copy()

translation_candidates_df = (
    translation_candidates_df
    .drop_duplicates(subset=["prompt_clean"])
    .reset_index(drop=True)
)

# Direction comes from the original dataset source.
translation_candidates_df.loc[
    translation_candidates_df["source"]
    == "EthioNLP:amharic_mt amh-eng",
    "direction"
] = "am->en"

translation_candidates_df.loc[
    translation_candidates_df["source"]
    == "EthioNLP:amharic_mt eng-amh",
    "direction"
] = "en->am"

translation_candidates_df[
    ["prompt", "task", "source",
     "direction", "source_output"]
].to_csv(
    f"{DATA_DIR}/translation_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)


# READING COMPREHENSION FILTER

reading_source_df = public_candidates_df[
    public_candidates_df["source"]
    == "EthioNLP:amharicqa"
].copy()

reading_source_df["prompt_clean"] = (
    reading_source_df["prompt"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

reading_patterns = (
    r"(አውድ.*መሰረት.*መልስ|"
    r"አውድ.*መልስ|"
    r"መረጃ.*በመጠቀም.*መልስ|"
    r"መረጃ.*ይመልከቱ|"
    r"ጽሑፍ.*በማጣቀስ.*መልስ|"
    r"ጽሁፍ.*በማጣቀስ.*መልስ|"
    r"ከሚከተለዉ.*አውድ|"
    r"ከዚህ.*በታች.*መረጃ.*ጥያቄ|"
    r"ከዚህ.*በታች.*ያለውን.*ጽሑፍ.*መልስ)"
)

bad_reading_patterns = (
    r"(ጥያቄ.*አውጣ|"
    r"ጥያቄ.*ፍጠር|"
    r"እውነት.*ሀሰት.*ጥያቄ|"
    r"እውነት.*ሀሰት.*መልስ.*ያለዉ.*ጥያቄ|"
    r"multiple.?choice|"
    r"አማራጭ.*ጥያቄ)"
)

is_reading = reading_source_df[
    "prompt_clean"
].str.contains(
    reading_patterns,
    regex=True,
    na=False
)

is_bad_reading = reading_source_df[
    "prompt_clean"
].str.contains(
    bad_reading_patterns,
    regex=True,
    na=False
)

reading_candidates_df = reading_source_df[
    is_reading & ~is_bad_reading
].copy()

reading_candidates_df = (
    reading_candidates_df
    .drop_duplicates(subset=["prompt_clean"])
    .reset_index(drop=True)
)

reading_candidates_df[
    ["prompt", "task", "source", "source_output"]
].to_csv(
    f"{DATA_DIR}/reading_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)


# SAVE EVAL DUPLICATES / REMOVALS

removed_eval_df = eval_duplicates.copy()

removed_eval_df["task"] = ""
removed_eval_df["source"] = (
    "EthioNLP:" + removed_eval_df["datasource"]
)
removed_eval_df["reason"] = (
    "exact_eval_prompt_duplicate"
)

removed_eval_df[
    ["source_prompt", "task", "source", "reason"]
].rename(
    columns={"source_prompt": "prompt"}
).to_csv(
    f"{DATA_DIR}/removed_duplicates.csv",
    index=False,
    encoding="utf-8-sig"
)


# FINAL REPORT

print("\n" + "=" * 60)
print("DATASET FILTERING COMPLETE")
print("=" * 60)

print("\nEvaluation duplicates removed:",
      len(eval_duplicates))

print("\nClean candidate counts:")
print(
    "  Reading comprehension:",
    len(reading_candidates_df)
)

print(
    "  Translation:",
    len(translation_candidates_df)
)

print(
    "  Summarization:",
    len(summary_candidates_df)
)

print("\nTranslation directions:")
print(
    translation_candidates_df["direction"]
    .value_counts()
)

print("\nSaved files:")

for filename in [
    "eval_prompts_60.csv",
    "predictions_template.csv",
    "reading_candidates.csv",
    "translation_candidates.csv",
    "summarization_candidates.csv",
    "removed_duplicates.csv",
]:

    path = f"{DATA_DIR}/{filename}"

    print(
        f"  {filename}:",
        "OK" if os.path.exists(path) else "MISSING"
    )

print("\nData directory:", DATA_DIR)

## 3. Build the final 2,800-prompt training pool

The assignment requires 1,000–3,000 prompt/answer pairs. We first build a balanced **2,800-prompt pool**:

- 700 closed-book knowledge
- 700 instruction following / reading comprehension
- 700 translation
- 700 summarization

At this stage these are **prompts only**. The answers will later be written by the teacher model.

The final construction code also performs an exact/near-duplicate check against the 60 evaluation prompts and saves examples removed from the pool.


In [ ]:
# AmhaDistill FINAL TRAINING PROMPT CONSTRUCTION
# Target: 2,800 examples = 700 per task

import os
import re
import unicodedata
import random
import hashlib
from pathlib import Path

import pandas as pd
from difflib import SequenceMatcher
# 1. Paths

PROJECT_DIR = Path("/content/drive/MyDrive/amha-distill")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# 2. Normalization

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = unicodedata.normalize("NFC", x)
    x = x.lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x


def is_eval_duplicate(prompt, eval_norms, threshold=93):
    """
    Exact or very-close match against the 60 evaluation prompts.
    """
    p = normalize_text(prompt)

    if p in eval_norms:
        return True, "exact_eval_duplicate"

    # Evaluation set is only 60 rows, so this is cheap.
    for e in eval_norms:
        if SequenceMatcher(None, p, e).ratio() * 100 >= threshold:
            return True, "near_eval_duplicate"

    return False, None


# 3. Load evaluation prompts

eval_path = DATA_DIR / "eval_prompts_60.csv"
eval_df = pd.read_csv(eval_path)

eval_norms = set(eval_df["prompt"].map(normalize_text))

print("=" * 70)
print("EVALUATION SET")
print("=" * 70)
print("Rows:", len(eval_df))
print(eval_df["task"].value_counts())
print()


# 4. Load candidate pools

reading = pd.read_csv(DATA_DIR / "reading_candidates.csv")
translation = pd.read_csv(DATA_DIR / "translation_candidates.csv")
summarization = pd.read_csv(DATA_DIR / "summarization_candidates.csv")

print("=" * 70)
print("CANDIDATE POOLS")
print("=" * 70)
print("Reading:", len(reading))
print("Translation:", len(translation))
print("Summarization:", len(summarization))
print()


# 5. CLOSED BOOK KNOWLEDGE
#
# We create 140 knowledge concepts × 5 question formulations
# = 700 prompts.
#
# The questions are intentionally answerable without a passage.

knowledge_topics = [

    # Geography
    "ኢትዮጵያ",
    "ኬንያ",
    "ኡጋንዳ",
    "ታንዛኒያ",
    "ሩዋንዳ",
    "ቡሩንዲ",
    "ሱዳን",
    "ግብፅ",
    "ሶማሊያ",
    "ኤርትራ",
    "ደቡብ ሱዳን",
    "ናይጄሪያ",
    "ጋና",
    "ሴኔጋል",
    "ደቡብ አፍሪካ",
    "ሞሮኮ",
    "አልጄሪያ",
    "ቱኒዚያ",
    "ፈረንሳይ",
    "ጀርመን",

    # Science
    "ፎቶሲንተሲስ",
    "የስበት ኃይል",
    "የውሃ ዑደት",
    "የካርቦን ዑደት",
    "ዲኤንኤ",
    "አርኤንኤ",
    "ሴል",
    "አቶም",
    "ሞለኪውል",
    "ኤሌክትሮን",
    "ፕሮቶን",
    "ኒውትሮን",
    "የኤሌክትሪክ ፍሰት",
    "ሙቀት",
    "ብርሃን",
    "ድምፅ",
    "ጨረቃ",
    "ፀሐይ",
    "ማርስ",
    "ጁፒተር",

    # Computing
    "ኮምፒውተር",
    "ፕሮሰሰር",
    "RAM",
    "ROM",
    "ሃርድ ድራይቭ",
    "SSD",
    "ኦፕሬቲንግ ሲስተም",
    "ሊኑክስ",
    "ኢንተርኔት",
    "ድር",
    "IP አድራሻ",
    "MAC አድራሻ",
    "ዳታቤዝ",
    "አልጎርይዝም",
    "ፕሮግራሚንግ",
    "Python",
    "Java",
    "JavaScript",
    "API",
    "ክላውድ ኮምፒውቲንግ",

    # Mathematics
    "ቁጥር",
    "ኢንቲጀር",
    "ፕራይም ቁጥር",
    "ክፍልፋይ",
    "መቶኛ",
    "አልጀብራ",
    "ጂኦሜትሪ",
    "ካሬ",
    "ክብ",
    "ሶስት ማዕዘን",
    "አራት ማዕዘን",
    "ዲያሜትር",
    "ራዲየስ",
    "ፒ",
    "ዕድል",
    "አማካይ",
    "መካከለኛ",
    "ሞድ",
    "ግራፍ",
    "ስታቲስቲክስ",

    # History / society
    "የኢትዮጵያ ታሪክ",
    "የሮማ ግዛት",
    "የግብፅ ፒራሚዶች",
    "የህዳሴ ዘመን",
    "የኢንዱስትሪ አብዮት",
    "የፈረንሳይ አብዮት",
    "የአሜሪካ ነፃነት",
    "የመጀመሪያው ዓለም ጦርነት",
    "የሁለተኛው ዓለም ጦርነት",
    "የአፍሪካ ኅብረት",
    "የተባበሩት መንግሥታት",
    "የኢትዮጵያ ባህላዊ በዓላት",
    "ዜግነት",
    "ዴሞክራሲ",
    "ሕገ መንግሥት",
    "የሰብዓዊ መብቶች",
    "ዓለም አቀፍ ንግድ",
    "ኢኮኖሚ",
    "ግብር",
    "ባንክ",

    # Biology
    "ሰውነት",
    "ልብ",
    "ሳንባ",
    "አንጎል",
    "ደም",
    "ኩላሊት",
    "ጉበት",
    "የመከላከያ ሥርዓት",
    "ቫይረስ",
    "ባክቴሪያ",
    "እፅዋት",
    "ዛፍ",
    "ሥር",
    "ቅጠል",
    "አበባ",
    "ዘር",
    "ማህደረ ዘር",
    "ዝግመተ ለውጥ",
    "ምግብ ሰንሰለት",
    "ሥነ ምህዳር",

    # Language / culture
    "ቋንቋ",
    "ሰዋሰው",
    "ስም",
    "ግስ",
    "ቅጽል",
    "ግስ ተውሳክ",
    "የአማርኛ ፊደል",
    "ትርጉም",
    "ሥነ ጽሑፍ",
    "ግጥም",
    "ልቦለድ",
    "ድራማ",
    "ተረት",
    "ምሳሌ",
    "ባህል",
    "ሙዚቃ",
    "ስዕል",
    "ቲያትር",
    "ፊልም",
    "ጋዜጠኝነት",

    # Everyday / general knowledge
    "ትምህርት",
    "ጤና",
    "ንፅህና",
    "አመጋገብ",
    "ስፖርት",
    "እግር ኳስ",
    "ቅርጫት ኳስ",
    "ኦሎምፒክ",
    "መጓጓዣ",
    "አውሮፕላን",
    "መኪና",
    "ባቡር",
    "መርከብ",
    "ኃይል",
    "ፀሐይ ኃይል",
    "ንፋስ ኃይል",
    "የአየር ንብረት",
    "ዝናብ",
    "ድርቅ",
    "ጎርፍ",
]

# Keep exactly 140 concepts.
knowledge_topics = knowledge_topics[:140]

knowledge_templates = [
    "{topic} ምንድን ነው? በአጭሩ ግለጽ።",
    "{topic} ምን ማለት ነው? ዋናውን ትርጉም ግለጽ።",
    "{topic} ስለሚመለከተው ዋና ነገር ምንድን ነው?",
    "{topic} ለምን አስፈላጊ ነው? በአጭሩ መልስ።",
    "{topic} ከምን ጋር ይያያዛል? አጭር ማብራሪያ ስጥ።",
]

closed_book = []

for topic in knowledge_topics:
    for template in knowledge_templates:
        closed_book.append({
            "prompt": template.format(topic=topic),
            "task": "closed_book_knowledge",
            "source": "controlled_knowledge_topics",
        })

closed_book = pd.DataFrame(closed_book)

# If the list somehow contains fewer than 700 unique prompts,
# stop rather than silently creating bad duplicates.
closed_book["norm"] = closed_book["prompt"].map(normalize_text)
closed_book = closed_book.drop_duplicates("norm").drop(columns=["norm"])

print("Closed-book generated:", len(closed_book))


# 6. REMOVE EVAL DUPLICATES FROM CLOSED BOOK

removed = []

keep_rows = []

for _, row in closed_book.iterrows():
    dup, reason = is_eval_duplicate(row["prompt"], eval_norms)

    if dup:
        removed.append({
            "prompt": row["prompt"],
            "task": row["task"],
            "source": row["source"],
            "removal_reason": reason,
        })
    else:
        keep_rows.append(row)

closed_book = pd.DataFrame(keep_rows)

# We need exactly 700.
if len(closed_book) < 700:
    raise RuntimeError(
        f"Only {len(closed_book)} closed-book prompts survived. "
        "Need 700."
    )

closed_book = closed_book.sample(
    n=700,
    random_state=SEED
).reset_index(drop=True)


# 7. READING COMPREHENSION

reading = reading.copy()

# Remove exact duplicates internally.
reading["_norm"] = reading["prompt"].map(normalize_text)
reading = reading.drop_duplicates("_norm").drop(columns=["_norm"])

# Remove evaluation leakage.
reading_keep = []

for _, row in reading.iterrows():
    dup, reason = is_eval_duplicate(row["prompt"], eval_norms)

    if dup:
        removed.append({
            "prompt": row["prompt"],
            "task": "instruction_following",
            "source": row.get("source", "reading_candidates"),
            "removal_reason": reason,
        })
    else:
        reading_keep.append(row)

reading = pd.DataFrame(reading_keep)

print("Clean reading candidates:", len(reading))


# We have 466 genuine reading examples.
# Use all available genuine examples.
real_reading = reading.copy()

if len(real_reading) > 700:
    real_reading = real_reading.sample(
        n=700,
        random_state=SEED
    )


# 8. READING AUGMENTATION
#
# We need 234 additional reading examples.
#
# Instead of inventing unrelated passages, we use existing
# passage/question examples and create a different comprehension
# instruction around the same underlying passage. The teacher
# will generate the answer for the new prompt later.

needed_reading = 700 - len(real_reading)

if needed_reading > 0:

    augmentation_pool = real_reading.sample(
        n=needed_reading,
        random_state=SEED
    ).copy()

    augmented = []

    augmentation_templates = [
        "ከሚከተለው ጽሑፍ በመነሳት ዋናውን መልስ በአጭሩ ስጥ።\n\n",
        "የተሰጠውን ጽሑፍ በመመርመር ጥያቄውን በአጭሩ መልስ።\n\n",
        "በጽሑፉ ውስጥ በተገለጸው መረጃ ብቻ ተመስርተህ መልስ።\n\n",
        "ከጽሑፉ የሚገኘውን መረጃ ተጠቅመህ ትክክለኛውን መልስ ስጥ።\n\n",
        "ጽሑፉን በማንበብ ለተጠየቀው ጥያቄ አጭር መልስ ስጥ።\n\n",
    ]

    for i, (_, row) in enumerate(augmentation_pool.iterrows()):

        original_prompt = str(row["prompt"]).strip()

        # Different instruction prefix creates a new training example.
        prefix = augmentation_templates[i % len(augmentation_templates)]

        new_prompt = prefix + original_prompt

        augmented.append({
            "prompt": new_prompt,
            "task": "instruction_following",
            "source": "reading_candidates_augmented",
        })

    augmented = pd.DataFrame(augmented)

    reading_final = pd.concat(
        [
            real_reading[["prompt"]].assign(
                task="instruction_following",
                source=real_reading["source"]
            ),
            augmented
        ],
        ignore_index=True
    )

else:
    reading_final = real_reading[["prompt"]].assign(
        task="instruction_following",
        source=real_reading["source"]
    )


reading_final = reading_final.head(700).reset_index(drop=True)

print("Final reading examples:", len(reading_final))


# 9. TRANSLATION

translation = translation.copy()

translation["_norm"] = translation["prompt"].map(normalize_text)
translation = translation.drop_duplicates("_norm").drop(columns=["_norm"])

translation_keep = []

for _, row in translation.iterrows():
    dup, reason = is_eval_duplicate(row["prompt"], eval_norms)

    if dup:
        removed.append({
            "prompt": row["prompt"],
            "task": "translation",
            "source": row.get("source", "translation_candidates"),
            "removal_reason": reason,
        })
    else:
        translation_keep.append(row)

translation = pd.DataFrame(translation_keep)

# Split by direction.
amh_en = translation[
    translation["direction"].astype(str).str.lower().str.contains("am")
    & translation["direction"].astype(str).str.lower().str.contains("en")
].copy()

# More reliable direction handling using source labels where possible.
amh_en_mask = translation["source"].astype(str).str.contains(
    "amh-eng", case=False, na=False
)

eng_amh_mask = translation["source"].astype(str).str.contains(
    "eng-amh", case=False, na=False
)

if amh_en_mask.sum() >= 350 and eng_amh_mask.sum() >= 350:
    amh_en = translation[amh_en_mask].copy()
    eng_amh = translation[eng_amh_mask].copy()
else:
    # Fall back to direction column.
    direction_norm = translation["direction"].astype(str).str.lower()

    amh_en = translation[
        direction_norm.str.contains("am") &
        direction_norm.str.contains("en")
    ].copy()

    eng_amh = translation[
        direction_norm.str.contains("en") &
        direction_norm.str.contains("am")
    ].copy()

print("Translation AM→EN candidates:", len(amh_en))
print("Translation EN→AM candidates:", len(eng_amh))

if len(amh_en) < 350 or len(eng_amh) < 350:
    raise RuntimeError(
        "Not enough clean translation candidates for the required "
        "350 AM→EN + 350 EN→AM."
    )

amh_en = amh_en.sample(n=350, random_state=SEED)
eng_amh = eng_amh.sample(n=350, random_state=SEED + 1)

translation_final = pd.concat(
    [
        amh_en[["prompt", "task", "source", "direction"]],
        eng_amh[["prompt", "task", "source", "direction"]],
    ],
    ignore_index=True
)

translation_final["task"] = "translation"

print("Final translation examples:", len(translation_final))


# 10. SUMMARIZATION

summarization = summarization.copy()

print("Starting summarization processing...")
print("Candidate pool:", len(summarization))

# Remove internal exact duplicates first.
summarization["_norm"] = summarization["prompt"].map(normalize_text)
summarization = summarization.drop_duplicates("_norm").drop(columns=["_norm"])

print("After internal dedup:", len(summarization))

# FAST EVAL LEAKAGE CHECK
#
# First remove exact matches from the entire pool.
# This is very fast.

summarization["_norm"] = summarization["prompt"].map(normalize_text)

exact_eval_mask = summarization["_norm"].isin(eval_norms)

exact_removed = summarization[exact_eval_mask]

for _, row in exact_removed.iterrows():
    removed.append({
        "prompt": row["prompt"],
        "task": "summarization",
        "source": row.get("source", "summarization_candidates"),
        "removal_reason": "exact_eval_duplicate",
    })

summarization = summarization[~exact_eval_mask].copy()
summarization = summarization.drop(columns=["_norm"])

print("After exact eval removal:", len(summarization))

# Randomly select a manageable screening pool.
#
# We need only 700 final examples.
# Take 1,000 candidates and perform fuzzy checking only
# on those.

screen_size = min(1000, len(summarization))

screen = summarization.sample(
    n=screen_size,
    random_state=SEED
).copy()

print("Fuzzy-screening:", len(screen), "candidates...")

screen_keep = []

for _, row in screen.iterrows():

    dup, reason = is_eval_duplicate(
        row["prompt"],
        eval_norms,
        threshold=93
    )

    if dup:
        removed.append({
            "prompt": row["prompt"],
            "task": "summarization",
            "source": row.get("source", "summarization_candidates"),
            "removal_reason": reason,
        })
    else:
        screen_keep.append(row)

screen_keep = pd.DataFrame(screen_keep)

print("Fuzzy-screened clean:", len(screen_keep))

# We need 700.
# If fuzzy screening accidentally removes too many,
# take additional candidates using exact matching only.

if len(screen_keep) < 700:

    remaining = summarization[
        ~summarization["prompt"].isin(screen_keep["prompt"])
    ].copy()

    needed = 700 - len(screen_keep)

    # Exact eval duplicates were already removed from the
    # complete pool, so selecting here is safe.
    additional = remaining.sample(
        n=needed,
        random_state=SEED + 10
    )

    screen_keep = pd.concat(
        [screen_keep, additional],
        ignore_index=True
    )

summarization_final = screen_keep.sample(
    n=700,
    random_state=SEED
).copy()

summarization_final["task"] = "summarization"

summarization_final = summarization_final[
    ["prompt", "task", "source"]
].reset_index(drop=True)

print("Final summarization examples:", len(summarization_final))


# 11. STANDARDIZE CLOSED BOOK

closed_book_final = closed_book[
    ["prompt", "task", "source"]
].copy()


# 12. COMBINE ALL TASKS

final_df = pd.concat(
    [
        closed_book_final,
        reading_final[["prompt", "task", "source"]],
        translation_final[["prompt", "task", "source"]],
        summarization_final[["prompt", "task", "source"]],
    ],
    ignore_index=True
)

# Shuffle once.
final_df = final_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# Add stable ID.
final_df.insert(
    0,
    "example_id",
    [f"TR{i:04d}" for i in range(1, len(final_df) + 1)]
)


# 13. FINAL INTERNAL DUPLICATE CHECK

final_df["_norm"] = final_df["prompt"].map(normalize_text)

duplicate_count = final_df["_norm"].duplicated().sum()

if duplicate_count > 0:
    print("WARNING:", duplicate_count, "internal duplicate prompts found.")

    final_df = final_df.drop_duplicates(
        "_norm",
        keep="first"
    ).reset_index(drop=True)

final_df = final_df.drop(columns=["_norm"])


# 14. CHECK TASK BALANCE

counts = final_df["task"].value_counts()

print("\n" + "=" * 70)
print("FINAL DATASET COUNTS")
print("=" * 70)
print(counts)
print("TOTAL:", len(final_df))

expected = {
    "closed_book_knowledge": 700,
    "instruction_following": 700,
    "translation": 700,
    "summarization": 700,
}

for task, expected_count in expected.items():
    actual = counts.get(task, 0)

    if actual != expected_count:
        raise RuntimeError(
            f"{task}: expected {expected_count}, got {actual}"
        )

assert len(final_df) == 2800


# 15. SAVE EVERYTHING IMMEDIATELY

final_path = DATA_DIR / "training_prompts_2800.csv"

closed_path = DATA_DIR / "closed_book_700.csv"
reading_path = DATA_DIR / "reading_700.csv"
translation_path = DATA_DIR / "translation_700.csv"
summary_path = DATA_DIR / "summarization_700.csv"
removed_path = DATA_DIR / "removed_prompt_examples.csv"

final_df.to_csv(final_path, index=False, encoding="utf-8-sig")

closed_book_final.to_csv(
    closed_path,
    index=False,
    encoding="utf-8-sig"
)

reading_final[["prompt", "task", "source"]].to_csv(
    reading_path,
    index=False,
    encoding="utf-8-sig"
)

translation_final.to_csv(
    translation_path,
    index=False,
    encoding="utf-8-sig"
)

summarization_final.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(removed).to_csv(
    removed_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("SAVED SUCCESSFULLY")
print("=" * 70)

for p in [
    final_path,
    closed_path,
    reading_path,
    translation_path,
    summary_path,
    removed_path,
]:
    print(f"✓ {p}")

print("\nDONE — 2,800 training prompts are now persisted in Google Drive.")

In [ ]:
# Audit the saved 2,800 prompt pool.

prompt_pool = pd.read_csv(DATA_DIR / "training_prompts_2800.csv")
eval_df = pd.read_csv(DATA_DIR / "eval_prompts_60.csv")

print("=" * 70)
print("PROMPT POOL AUDIT")
print("=" * 70)
print("Rows:", len(prompt_pool))
print("Columns:", prompt_pool.columns.tolist())
print("\nTask distribution:")
print(prompt_pool["task"].value_counts())

def normalize_for_check(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFC", str(x))
    return re.sub(r"\s+", " ", x).strip().lower()

eval_norms = set(eval_df["prompt"].map(normalize_for_check))
train_norms = set(prompt_pool["prompt"].map(normalize_for_check))

exact_overlap = train_norms.intersection(eval_norms)

print("\nExact evaluation-prompt overlap:", len(exact_overlap))

if exact_overlap:
    raise RuntimeError("Evaluation leakage detected in the prompt pool.")


## 4. Teacher model: Gemini

The teacher generates the answers used as supervision.

This is **response-level distillation**:

```text
prompt ──> teacher ──> teacher response
                         ↓
                    student learns
```

We do **not** copy the teacher's hidden states or probability/logit distribution. The student simply learns from the teacher-written response tokens.

The teacher used in the final generation experiment was:

`gemini-3.8-flash`

The generation code is checkpointed so that an interrupted run can resume from the existing CSV.


In [ ]:
from google.colab import userdata
import requests
import time

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("Gemini API key available:", bool(GEMINI_API_KEY))

TEACHER_MODEL = "gemini-3.8-flash"
PROMPTS_PATH = DATA_DIR / "training_prompts_2800.csv"
TEACHER_OUTPUT = DATA_DIR / "teacher_responses_2800.csv"
FINAL_TRAINING = DATA_DIR / "final_training_1144.csv"

# Set this to True only if you intentionally want to make/resume
# teacher API requests. The saved final dataset is used by default.
RUN_TEACHER_GENERATION = False


In [ ]:
# ============================================================
# GEMINI TEACHER TEST
# ============================================================

!pip install -q -U google-genai

from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("Gemini API key available:", bool(GEMINI_API_KEY))

client = genai.Client(api_key=GEMINI_API_KEY)

TEACHER_MODEL = "gemini-3.8-flash"

prompt = """You are a high-quality multilingual teacher model helping create
an Amharic knowledge-distillation dataset.

Translate the following English sentence into natural, accurate Amharic.
Return ONLY the translation. Do not explain your answer.

English:
Machine learning allows computers to learn patterns from data.
"""

response = client.models.generate_content(
    model=TEACHER_MODEL,
    contents=prompt
)

print("\n" + "=" * 70)
print("GEMINI TEACHER RESPONSE")
print("=" * 70)
print(response.text)
print("=" * 70)

In [ ]:
import os
import time
import requests
import pandas as pd
from google.colab import userdata

# ============================================================
# CONFIG
# ============================================================

MODEL = "gemini-3.8-flash"

DATA_DIR = "/content/drive/MyDrive/amha-distill/data"
PROMPTS_PATH = f"{DATA_DIR}/training_prompts_2800.csv"
OUTPUT_PATH = f"{DATA_DIR}/teacher_responses_2800.csv"

API_KEY = userdata.get("GEMINI_API_KEY")

URL = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"models/{MODEL}:generateContent"
)

# ============================================================
# LOAD EXISTING DATA
# ============================================================

prompts = pd.read_csv(PROMPTS_PATH)

responses = pd.read_csv(OUTPUT_PATH)

responses = responses.drop_duplicates(
    subset=["example_id"],
    keep="last"
).reset_index(drop=True)

done_ids = set(responses["example_id"].astype(str))

remaining = prompts[
    ~prompts["example_id"].astype(str).isin(done_ids)
].copy()

print("=" * 70)
print("RESUMING TEACHER GENERATION")
print("=" * 70)
print("Teacher:", MODEL)
print("Total prompts:", len(prompts))
print("Already completed:", len(responses))
print("Remaining:", len(remaining))
print("=" * 70)


# ============================================================
# BUILD TASK SPECIFIC PROMPT
# ============================================================

def build_teacher_prompt(row):

    task = str(row["task"])
    prompt = str(row["prompt"])

    if task == "translation":

        instruction = """
You are a high-quality multilingual translation teacher.

Translate the user's text exactly as requested.

Rules:
- Output ONLY the translation.
- Do NOT explain the translation.
- Do NOT add introductions, labels, notes, or commentary.
- Preserve the meaning of the original.
- Preserve names, numbers, and important details.
"""

    elif task == "summarization":

        instruction = """
You are an expert Amharic summarization teacher.

Summarize the provided text according to the user's request.

Rules:
- Output ONLY the requested summary.
- Do NOT explain your reasoning.
- Do NOT add introductions, labels, or commentary.
- Keep the important information.
- Do not invent facts that are absent from the source.
"""

    elif task == "instruction_following":

        instruction = """
You are an expert Amharic reading-comprehension teacher.

Answer the user's question using ONLY the information contained
in the provided passage.

Rules:
- Answer the question directly.
- Keep the answer concise unless the prompt asks for detail.
- Do NOT invent information that is not supported by the passage.
- Output ONLY the answer.
"""

    elif task == "closed_book_knowledge":

        instruction = """
You are an expert multilingual knowledge teacher.

Answer the user's question accurately.

Rules:
- Answer directly.
- Be concise unless the question requires explanation.
- Do NOT mention that you are an AI.
- Do NOT add unnecessary commentary.
- Output ONLY the answer.
"""

    else:

        instruction = """
Answer the user's request accurately and directly.
Output ONLY the requested answer.
"""

    return instruction + "\n\nUSER REQUEST:\n" + prompt


# ============================================================
# ONE GEMINI REQUEST
# ============================================================

def generate_one(text, max_retries=3):

    payload = {
        "contents": [
            {
                "parts": [
                    {
                        "text": text
                    }
                ]
            }
        ],
        "generationConfig": {
            "thinkingConfig": {
                "thinkingLevel": "low"
            },
            "temperature": 0.2,
            "maxOutputTokens": 512
        }
    }

    for attempt in range(1, max_retries + 1):

        try:

            r = requests.post(
                URL,
                params={"key": API_KEY},
                headers={
                    "Content-Type": "application/json"
                },
                json=payload,
                timeout=60
            )

            # SUCCESS

            if r.status_code == 200:

                data = r.json()

                candidates = data.get("candidates", [])

                if not candidates:
                    return None, "NO_CANDIDATE"

                content = candidates[0].get(
                    "content", {}
                )

                parts = content.get("parts", [])

                if not parts:
                    return None, "NO_TEXT"

                answer = parts[0].get(
                    "text", ""
                ).strip()

                if not answer:
                    return None, "EMPTY"

                return answer, "OK"

            # QUOTA

            if r.status_code == 429:

                try:
                    data = r.json()
                    message = data.get(
                        "error", {}
                    ).get(
                        "message",
                        r.text
                    )
                except:
                    message = r.text

                print("\n" + "=" * 70)
                print("🚨 QUOTA / RATE LIMIT")
                print("=" * 70)
                print(message[:1500])
                print("=" * 70)

                return None, "QUOTA"

            # TEMPORARY SERVER ERROR

            if r.status_code in [500, 502, 503, 504]:

                print(
                    f"   ⚠️ HTTP {r.status_code} "
                    f"(attempt {attempt}/{max_retries})"
                )

                if attempt < max_retries:

                    wait = 15 * attempt

                    print(
                        f"   Waiting {wait}s..."
                    )

                    time.sleep(wait)

                    continue

                return None, "SERVER_ERROR"

            # OTHER ERROR

            try:
                data = r.json()
                message = data.get(
                    "error", {}
                ).get(
                    "message",
                    r.text
                )
            except:
                message = r.text

            print(
                f"   ❌ HTTP {r.status_code}: "
                f"{str(message)[:500]}"
            )

            return None, f"HTTP_{r.status_code}"

        except requests.exceptions.Timeout:

            print(
                f"   ⏱️ Timeout "
                f"(attempt {attempt}/{max_retries})"
            )

            if attempt < max_retries:
                time.sleep(10 * attempt)
            else:
                return None, "TIMEOUT"

        except requests.exceptions.RequestException as e:

            print(
                f"   ⚠️ Network error: "
                f"{str(e)[:300]}"
            )

            if attempt < max_retries:
                time.sleep(10 * attempt)
            else:
                return None, "NETWORK_ERROR"

    return None, "FAILED"


# ============================================================
# GENERATION LOOP
# ============================================================

successful_since_save = 0

for position, (_, row) in enumerate(
    remaining.iterrows(),
    start=1
):

    example_id = str(row["example_id"])

    print(
        f"\n[{position}/{len(remaining)}] "
        f"Generating {example_id}..."
    )

    teacher_prompt = build_teacher_prompt(row)

    answer, status = generate_one(
        teacher_prompt
    )

    # QUOTA → STOP CLEANLY

    if status == "QUOTA":

        print("\n🛑 Quota reached.")
        print("All progress has been saved.")
        break

    # FAILED → SKIP FOR NOW

    if answer is None:

        print(
            f"   ❌ Failed: {status}"
        )

        continue

    # ADD RESPONSE

    new_row = pd.DataFrame([{
        "example_id": example_id,
        "prompt": row["prompt"],
        "teacher_response": answer,
        "task": row["task"],
        "source": row["source"]
    }])

    responses = pd.concat(
        [responses, new_row],
        ignore_index=True
    )

    print(
        f"   ✓ Saved "
        f"({len(responses)}/2800)"
    )

    successful_since_save += 1

    # CHECKPOINT

    if successful_since_save >= 5:

        responses = responses.drop_duplicates(
            subset=["example_id"],
            keep="last"
        ).reset_index(drop=True)

        responses.to_csv(
            OUTPUT_PATH,
            index=False
        )

        successful_since_save = 0

        print(
            f"   💾 CHECKPOINT: "
            f"{len(responses)}/2800"
        )

    # Small delay
    time.sleep(1)


# ============================================================
# FINAL SAVE
# ============================================================

responses = responses.drop_duplicates(
    subset=["example_id"],
    keep="last"
).reset_index(drop=True)

responses.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("GENERATION STOPPED / COMPLETED")
print("=" * 70)
print("Saved:", len(responses))
print("Remaining:", 2800 - len(responses))
print("Output:", OUTPUT_PATH)
print("=" * 70)

In [ ]:
import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/amha-distill/data"

teacher_path = f"{DATA_DIR}/teacher_responses_2800.csv"
final_path = f"{DATA_DIR}/final_training_1144.csv"

df = pd.read_csv(teacher_path)

# Keep only complete teacher examples
df = df.dropna(subset=["prompt", "teacher_response"])

# Remove duplicate IDs
df = df.drop_duplicates(subset=["example_id"], keep="last")

# Final assignment columns
final_df = df[
    ["prompt", "teacher_response", "task", "source"]
].copy()

final_df.to_csv(final_path, index=False)

print("=" * 60)
print("FINAL TRAINING DATASET")
print("=" * 60)
print("Examples:", len(final_df))
print()
print(final_df["task"].value_counts())
print()
print("Saved:", final_path)
print("=" * 60)

## 5. Final teacher-response dataset audit

The completed experiment contains **1,144** teacher-generated examples.

This audit is important because the final result depends not only on the LoRA code but also on the quality of the teacher responses.


In [ ]:
final_df = pd.read_csv(FINAL_TRAINING)

final_df["prompt"] = final_df["prompt"].fillna("").astype(str)
final_df["teacher_response"] = final_df["teacher_response"].fillna("").astype(str)

print("=" * 70)
print("FINAL TRAINING DATASET AUDIT")
print("=" * 70)
print("Rows:", len(final_df))
print("\nTasks:")
print(final_df["task"].value_counts())

print("\nMissing responses:", final_df["teacher_response"].isna().sum())
print("Empty responses:", (final_df["teacher_response"].str.strip() == "").sum())

word_lengths = final_df["teacher_response"].str.split().str.len()

print("\nTeacher-response word statistics:")
print(word_lengths.describe())

print(
    "\nVery short responses (<10 words):",
    (word_lengths < 10).sum(),
    "/",
    len(final_df)
)

print(
    "Responses >256 words:",
    (word_lengths > 256).sum()
)


## 6. Student model and LoRA fine-tuning

Student:

`Qwen/Qwen2.5-0.5B-Instruct`

The student has its own Qwen tokenizer.

We use **LoRA (Low-Rank Adaptation)** instead of full fine-tuning. LoRA freezes the pretrained model and learns small low-rank update matrices for selected attention projections.

Configuration used in the completed experiment:

- rank `r=8`
- `alpha=16`
- dropout `0.05`
- target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- learning rate `2e-4`
- 2 epochs
- batch size 1
- gradient accumulation 4
- FP16
- gradient checkpointing
- maximum sequence length 256

Most importantly, the loss is **response-only**: prompt tokens are assigned label `-100`, so they do not contribute to the language-model loss.


In [ ]:
import os
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_PATH = str(FINAL_TRAINING)
ADAPTER_DIR = str(ADAPTER_DIR)
MAX_LENGTH = 256

print("=" * 70)
print("STUDENT TRAINING")
print("=" * 70)
print("Student:", MODEL_NAME)
print("Examples:", len(final_df))
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nLoading student model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


def tokenize_example(example):
    user_text = str(example["prompt"])
    teacher_text = str(example["teacher_response"])

    messages = [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": teacher_text},
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    user_text_only = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": user_text}
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    user_tokens = tokenizer(
        user_text_only,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    user_length = min(
        len(user_tokens["input_ids"]),
        len(input_ids),
    )

    # negative 100 means "ignore this token when calculating loss".
    # Therefore only the teacher response portion contributes.
    labels = [-100] * user_length + input_ids[user_length:]
    labels = labels[:len(input_ids)]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


dataset = Dataset.from_pandas(
    final_df[["prompt", "teacher_response"]],
    preserve_index=False,
)

tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names,
    desc="Tokenizing",
)

print("Tokenized examples:", len(tokenized_dataset))


training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    fp16=True,
    bf16=False,
    dataloader_num_workers=0,
    gradient_checkpointing=True,
    optim="adamw_torch",
    remove_unused_columns=False,
)


def collate_fn(features):
    max_len = max(len(x["input_ids"]) for x in features)

    batch_input_ids = []
    batch_attention = []
    batch_labels = []

    for x in features:
        pad_len = max_len - len(x["input_ids"])

        batch_input_ids.append(
            x["input_ids"] + [tokenizer.pad_token_id] * pad_len
        )

        batch_attention.append(
            x["attention_mask"] + [0] * pad_len
        )

        batch_labels.append(
            x["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(batch_attention, dtype=torch.long),
        "labels": torch.tensor(batch_labels, dtype=torch.long),
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collate_fn,
)

print("\n" + "=" * 70)
print("STARTING LoRA TRAINING")
print("=" * 70)
print("GPU:", torch.cuda.get_device_name(0))
print("Examples:", len(tokenized_dataset))
print("Epochs:", 2)
print("Batch size:", 1)
print("Gradient accumulation:", 4)
print("=" * 70)

train_result = trainer.train()

print("\nSaving LoRA adapter...")

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print("Adapter:", ADAPTER_DIR)
print("Training loss:", train_result.training_loss)
print("=" * 70)


## 7. Evaluation: base vs distilled student

The same 60 held-out prompts are used for both models.

Required evaluation settings:

- greedy decoding (`do_sample=False`)
- `max_new_tokens=256`
- batch size 1
- same GPU
- base model and LoRA-distilled model
- raw outputs saved without modification

The output CSV is compatible with the assignment's prediction template.


In [ ]:
import time
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

EVAL_PATH = DATA_DIR / "eval_prompts_60.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "predictions_qwen25_05b_lora.csv"

eval_df = pd.read_csv(EVAL_PATH)

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to("cuda")
base_model.eval()

distilled_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to("cuda")

distilled_model = PeftModel.from_pretrained(
    distilled_base,
    str(ADAPTER_DIR),
).to("cuda")
distilled_model.eval()


def generate_one(model, prompt):
    messages = [{"role": "user", "content": str(prompt)}]

    text = base_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = base_tokenizer(
        text,
        return_tensors="pt",
    ).to("cuda")

    input_len = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=base_tokenizer.pad_token_id,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    seconds = time.perf_counter() - start

    new_tokens = output.shape[1] - input_len

    generated = output[0][input_len:]

    answer = base_tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer, seconds, int(new_tokens)


rows = []

print("=" * 70)
print("EVALUATION")
print("=" * 70)
print("Prompts:", len(eval_df))
print("GPU:", torch.cuda.get_device_name(0))
print("=" * 70)

for _, row in eval_df.iterrows():

    prompt = row["prompt"]

    base_output, base_seconds, base_tokens = generate_one(
        base_model,
        prompt,
    )

    distilled_output, distilled_seconds, distilled_tokens = generate_one(
        distilled_model,
        prompt,
    )

    rows.append({
        "eval_id": row["eval_id"],
        "task": row["task"],
        "base_output": base_output,
        "distilled_output": distilled_output,
        "base_seconds": base_seconds,
        "base_new_tokens": base_tokens,
        "distilled_seconds": distilled_seconds,
        "distilled_new_tokens": distilled_tokens,
        "teacher_output": "",
    })

    print(
        row["eval_id"],
        "|",
        row["task"],
        "| base:",
        round(base_seconds, 3),
        "s | distilled:",
        round(distilled_seconds, 3),
        "s",
    )

predictions_df = pd.DataFrame(rows)

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved:", PREDICTIONS_PATH)


## 8. Evaluation analysis

The following diagnostics were used after generation:

- average latency
- average generated-token count
- number of outputs hitting the 256-token limit
- consecutive-word repetition
- unique-word ratio

These metrics are diagnostic, not a substitute for human evaluation.


In [ ]:
pred = pd.read_csv(PREDICTIONS_PATH)

def repetition_score(text):
    words = str(text).split()
    if len(words) < 2:
        return 0.0

    repeated = sum(
        1 for a, b in zip(words, words[1:])
        if a == b
    )

    return repeated / max(1, len(words) - 1)


def unique_word_ratio(text):
    words = str(text).split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)


pred["base_repetition"] = pred["base_output"].map(repetition_score)
pred["distilled_repetition"] = pred["distilled_output"].map(repetition_score)

pred["base_unique_ratio"] = pred["base_output"].map(unique_word_ratio)
pred["distilled_unique_ratio"] = pred["distilled_output"].map(unique_word_ratio)

print("=" * 70)
print("RESULT SUMMARY")
print("=" * 70)

print(
    "Base average time:",
    round(pred["base_seconds"].mean(), 3),
    "seconds/answer"
)

print(
    "Distilled average time:",
    round(pred["distilled_seconds"].mean(), 3),
    "seconds/answer"
)

print(
    "Base average new tokens:",
    round(pred["base_new_tokens"].mean(), 1)
)

print(
    "Distilled average new tokens:",
    round(pred["distilled_new_tokens"].mean(), 1)
)

print(
    "Base 256-token hits:",
    int((pred["base_new_tokens"] >= 256).sum()),
    "/",
    len(pred)
)

print(
    "Distilled 256-token hits:",
    int((pred["distilled_new_tokens"] >= 256).sum()),
    "/",
    len(pred)
)

print(
    "Base repetition:",
    round(pred["base_repetition"].mean(), 4)
)

print(
    "Distilled repetition:",
    round(pred["distilled_repetition"].mean(), 4)
)

print(
    "Base unique-word ratio:",
    round(pred["base_unique_ratio"].mean(), 4)
)

print(
    "Distilled unique-word ratio:",
    round(pred["distilled_unique_ratio"].mean(), 4)
)

print("\nPer-task output counts:")
print(pred["task"].value_counts())


## 9. Completed experiment record

The completed run produced the following recorded results:

| Item | Result |
|---|---:|
| Teacher | `gemini-3.8-flash` |
| Final teacher-response examples | **1,144** |
| Student | `Qwen/Qwen2.5-0.5B-Instruct` |
| Fine-tuning | **LoRA** |
| Epochs | **2** |
| GPU | **Tesla T4** |
| Training loss | **3.034905020173613** |
| Evaluation prompts | **60** |
| Base average time | **6.485 s/answer** |
| Distilled average time | **8.325 s/answer** |
| Base average tokens | **187.7** |
| Distilled average tokens | **163.2** |
| Base 256-token hits | **31/60** |
| Distilled 256-token hits | **32/60** |
| Base repetition | **0.0981** |
| Distilled repetition | **0.2234** |
| Base unique-word ratio | **0.5958** |
| Distilled unique-word ratio | **0.6128** |

The qualitative result was weak: the distilled model showed substantially more repetition and several incoherent/unrelated answers. The final report should present this honestly rather than treating the experiment as a successful capability improvement.


## 10. Submission artifacts

Important files produced by the experiment:

- `data/eval_prompts_60.csv`
- `data/training_prompts_2800.csv`
- `data/final_training_1144.csv`
- `data/removed_prompt_examples.csv`
- `outputs/predictions_qwen25_05b_lora.csv`
- `adapter/adapter_qwen25_05b_lora/`

The very large raw public-data dump should generally stay outside GitHub if it exceeds repository limits. The cleaned candidate files and final training CSV are the important reproducibility artifacts.


In [ ]:
print("=" * 70)
print("SUBMISSION FILE CHECK")
print("=" * 70)

important = [
    DATA_DIR / "eval_prompts_60.csv",
    DATA_DIR / "training_prompts_2800.csv",
    DATA_DIR / "final_training_1144.csv",
    DATA_DIR / "removed_prompt_examples.csv",
    OUTPUT_DIR / "predictions_qwen25_05b_lora.csv",
]

for path in important:
    print(
        "✓" if path.exists() else "✗",
        path
    )

print("\nAdapter directory:")
print(
    "✓" if ADAPTER_DIR.exists() else "✗",
    ADAPTER_DIR
)
